# SIFLOW · run_3 · MDLM eval + figures (fills Table 2)

Evaluates the SIFLOW student (Gen-PPL, MAUVE, LAMBADA, diversity, throughput across k=1,2,4,8), the MDLM teacher step-curve, AR GPT-2, and (optionally) SDTT@8; then builds the straightness/Pareto/entropy figures and auto-fills the LaTeX tables.

**Hardware:** single A100-80GB, <12h. All artifacts are written to Google Drive so the next notebook resumes.

**Needs from previous notebooks:** run_2 (ckpt/mdlm), run_1 (val tokens)

In [ ]:
# --- 1. Clone + install (run once per session) ---
REPO_URL = "https://github.com/kagtgi/siflow.git"   # <-- edit to your fork if needed
import os
if not os.path.isdir("siflow"):
    !git clone $REPO_URL siflow
%cd siflow
!git pull -q
!pip -q install -e .
!pip -q install -r requirements-colab.txt
print("setup done")

In [ ]:
# --- 2. Mount Drive + set artifact base (all sessions share this) ---
from siflow.utils import drive
drive.mount()
import os
os.environ["SIFLOW_BASE"] = "/content/drive/MyDrive/siflow"
BASE = drive.base_dir()
print("artifacts ->", BASE)

In [ ]:
# SIFLOW student (k sweep + straightness + entropy analyses)
!python scripts/evaluate.py --config siflow/config/mdlm.yaml --system siflow \
    --ckpt-dir {BASE}/ckpt/mdlm --ref-tokens {BASE}/data/owt_gpt2_val.npy \
    --n-samples 1000 --out results/mdlm.json

In [ ]:
# MDLM teacher step-curve baseline
!python scripts/evaluate.py --config siflow/config/mdlm.yaml --system teacher \
    --steps 8 32 64 1024 --ref-tokens {BASE}/data/owt_gpt2_val.npy \
    --n-samples 1000 --out results/mdlm_teacher.json

In [ ]:
# AR GPT-2 reference
!python scripts/evaluate.py --config siflow/config/mdlm.yaml --system ar --ar-model gpt2 \
    --ref-tokens {BASE}/data/owt_gpt2_val.npy --n-samples 1000 --out results/ar_gpt2.json

In [ ]:
# (optional) SDTT@8 baseline -- same GPT-2 tokenizer / MDLM arch, directly comparable.
# Skips gracefully if the package / checkpoint is unavailable.
try:
    !pip -q install git+https://github.com/jdeschena/sdtt.git
    import torch, json
    from sdtt import load_small_student
    from siflow.eval.gen_ppl import GPT2Scorer, decode_ids
    from siflow.eval.diversity import diversity_metrics
    from transformers import AutoTokenizer
    m = load_small_student(loss="kld", round=7).cuda().eval()
    tok = AutoTokenizer.from_pretrained("gpt2")
    texts = []
    while len(texts) < 1000:
        s = m.sample(n_samples=64, num_steps=8, seq_len=256)
        texts += decode_ids(s if torch.is_tensor(s) else torch.tensor(s), tok)
    scorer = GPT2Scorer("gpt2-large")
    res = {"run_id": "sdtt", "method": "SDTT", "source": "reproduced",
           "metrics": {"steps=8": {**scorer.perplexity(texts), **diversity_metrics(texts), "nfe": 8}}}
    json.dump(res, open("results/sdtt.json", "w"), indent=2)
    print("SDTT done")
except Exception as e:
    print("SDTT baseline skipped:", e)

In [ ]:
# Figures + auto-filled tables
!python scripts/make_figures.py --results results --train-log {BASE}/ckpt/mdlm/train_log.jsonl
!python scripts/make_tables.py --results results
print(open("paper/tables_auto.tex").read()[:1500])

In [ ]:
# --- Save outputs to Drive (so the next notebook can resume) ---
!cp -r results {BASE}/ 2>/dev/null || true
!cp -r paper/figures {BASE}/ 2>/dev/null || true
!cp -r paper/tables_auto.tex {BASE}/ 2>/dev/null || true
print('saved to', BASE)